# Lesson 1.3 — The Twist
**Module 6 · Unit 1 · Lesson 3**

Verifies: (1) the rigid-body velocity field $v_P=v+\omega\times r$; (2) the rigid-body constraint (no stretching); (3) reference-point change leaves the field invariant.

Convention D-057: $\xi=[v;\omega]$ (linear on top).

In [1]:
import numpy as np
checks=[]
def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])

v=np.array([1.0,0,0]); omega=np.array([0,0,2.0])
xi=np.hstack([v,omega])                      # twist, linear on top (D-057)
def point_velocity(xi, r, ref=np.zeros(3)):
    v,omega=xi[:3],xi[3:]
    return v + np.cross(omega, np.asarray(r)-ref)

# worked example: point at (0,0.5,0) is instantaneously at rest
print("v_P at (0,0.5,0):", point_velocity(xi,[0,0.5,0]))
checks.append(np.allclose(point_velocity(xi,[0,0.5,0]), 0))

v_P at (0,0.5,0): [0. 0. 0.]


## Rigid-body constraint: relative velocity has no component along the link

In [2]:
rng=np.random.default_rng(0)
ok=True
for _ in range(200):
    P,Q=rng.normal(size=3),rng.normal(size=3)
    rel=point_velocity(xi,P)-point_velocity(xi,Q)
    if abs(rel@(P-Q))>1e-12: ok=False
checks.append(ok)
print("rigid-body constraint holds for 200 random point pairs:", ok)

rigid-body constraint holds for 200 random point pairs: True


## Reference-point change leaves the physical field unchanged

In [3]:
# express the same twist about a new reference point O' = (0,1,0)
Oprime=np.array([0,1.0,0])
v_Oprime=v+np.cross(omega, Oprime-np.zeros(3))   # v shifts by omega x d
xi2=np.hstack([v_Oprime, omega])
# velocity of a test point must match whichever reference we use
Ptest=np.array([0.3,-0.7,0.2])
a=point_velocity(xi, Ptest, ref=np.zeros(3))
b=point_velocity(xi2, Ptest, ref=Oprime)
print("via O :",a,"\nvia O':",b)
checks.append(np.allclose(a,b))

via O : [2.4 0.6 0. ] 
via O': [2.4 0.6 0. ]


In [4]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
